In [7]:
import pandas as pd
import sys, os
from pathlib import Path

import os
import math
sys.path.append(str(Path(os.getcwd()).parent))
from utils.wrappers import measure_time_and_space, measure_time
from utils.data_structures import RollingMeanArray, UpwardsDownwardsArray

### Getting list of filepaths

In [8]:
@measure_time_and_space
def select_target_csvs(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = {
        "MSFT.csv",
        "NVDA.csv",
        "AAPL.csv",
        "GOOGL.csv",
        "AMZN.csv",
        "META.csv",
        "TSLA.csv",
    }

    selected = []
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file() and entry.name in target_files:
                selected.append(entry.path)
    return selected

@measure_time_and_space
def select_target_csvs_nonrecursive(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = [
        "MSFT.csv",
        "NVDA.csv",
        "AAPL.csv",
        "GOOGL.csv",
        "AMZN.csv",
        "META.csv",
        "TSLA.csv",
        ]
    

    selected_paths = []

    # Efficiently iterate through directory (not recursive)
    for filename in os.listdir(directory):
        if filename in target_files:
            selected_paths.append(os.path.join(directory, filename))

    return selected_paths

In [9]:
select_target_csvs(Path.cwd() / "csv")
select_target_csvs_nonrecursive(Path.cwd() / "csv")



[select_target_csvs] Time elapsed: 0.000990 seconds
[select_target_csvs] Peak memory: 3.24 KB
[select_target_csvs_nonrecursive] Time elapsed: 0.000567 seconds
[select_target_csvs_nonrecursive] Peak memory: 38.16 KB


['c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\AAPL.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\AMZN.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\GOOGL.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\META.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\MSFT.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\NVDA.csv',
 'c:\\Users\\Admin\\.Projects\\Programming Fundamentals\\SIT_ICT1002\\data\\csv\\TSLA.csv']

In [10]:
file_paths = select_target_csvs(Path.cwd() / "csv")

[select_target_csvs] Time elapsed: 0.001166 seconds
[select_target_csvs] Peak memory: 3.24 KB


### Reading the selected csv files in a efficient manner

In [ ]:
@measure_time_and_space

def read_filtered_csvs(file_paths, chunksize=100000, min_year = 2025, max_year = 2025):
    """
    Reads multiple CSV files in chunks, filters rows with Date.year > 2022,
    and returns a single combined DataFrame.
    """
    combined_chunks = []  # list to store filtered chunks
    hloc_cols = ["High", "Low", "Open", "Close"]
    converters = {col: lambda x: round(float(x)) for col in hloc_cols}

    for path in file_paths:
        for chunk in pd.read_csv(path, parse_dates=['Date'], chunksize=chunksize, usecols= lambda x: x not in ["Adj Close"], converters=converters):
            filtered_chunk = chunk[(chunk['Date'].dt.year >= min_year) & (chunk['Date'].dt.year <= max_year)] # Filter rows where year > 2024
            if not filtered_chunk.empty:
                combined_chunks.append(filtered_chunk)

    # Concatenate all filtered chunks into a single DataFrame
    combined_df = pd.concat(combined_chunks, ignore_index=True) 
    return combined_df

combined_df = read_filtered_csvs(file_paths).drop_duplicates().reset_index(drop=True)

[read_filtered_csvs] Time elapsed: 0.504232 seconds
[read_filtered_csvs] Peak memory: 4096.14 KB


In [ ]:
combined_df.tail()

,Date,Ticker,Open,High,Low,Close,Volume
1343,2025-10-02,TSLA,471,471,436,436,137009000
1344,2025-10-03,TSLA,443,447,417,430,133188200
1345,2025-10-06,TSLA,441,454,437,453,85324900
1346,2025-10-07,TSLA,448,453,432,433,101798300
1347,2025-10-08,TSLA,438,438,425,433,21848536


In [ ]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1348 entries, 0 to 1347
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    1348 non-null   datetime64[ns]
 1   Ticker  1348 non-null   object        
 2   Open    1348 non-null   int64         
 3   High    1348 non-null   int64         
 4   Low     1348 non-null   int64         
 5   Close   1348 non-null   int64         
 6   Volume  1348 non-null   int64         
dtypes: datetime64[ns](1), int64(5), object(1)
memory usage: 73.8+ KB


## This section is experimental and is only intended to compare the space time complexity of various algorithms
### Actual feature engineering will be done below

In [ ]:
#Test Df so we can compare space time complexity between different implementations
test_df = combined_df[combined_df['Ticker'] == 'AAPL'].copy()

In [ ]:
test_df.head()

,Date,Ticker,Open,High,Low,Close,Volume
0,2025-01-02,AAPL,249,249,242,244,55740700
1,2025-01-03,AAPL,243,244,242,243,40244100
2,2025-01-06,AAPL,244,247,243,245,45045600
3,2025-01-07,AAPL,243,246,241,242,40856000
4,2025-01-08,AAPL,242,244,240,243,37628900


In [ ]:
@measure_time_and_space
def sma_pandas(df, window=20):
    """
    Calculate Simple Moving Average (SMA) for the 'close' column.
    """
    return df['Close'].rolling(window=window).mean()

sma_pandas(test_df, window=30)

[sma_pandas] Time elapsed: 0.000823 seconds
[sma_pandas] Peak memory: 11.47 KB


0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
          ...    
188    241.366667
189    242.333333
190    243.300000
191    244.266667
192    245.200000
Name: Close, Length: 193, dtype: float64

In [2]:
rma = RollingMeanArray(test_df['Close'],30)

In [12]:
@measure_time_and_space
def sma_naive():
    """
    Calculate Simple Moving Average (SMA) for the 'close' column using RollingMeanArray.naive_rolling_mean method, as a benchmark against the sma_rma method.
    """
    return pd.Series(rma.naive_rolling_mean())
# sma_naive()
test_df["SMA_30"] = sma_naive()

[sma_naive] Time elapsed: 0.000432 seconds
[sma_naive] Peak memory: 17.87 KB


In [13]:
@measure_time_and_space
def sma_rma():
    """
    Calculate Simple Moving Average (SMA) for the 'close' column using RollingMeanArray class.
    """
    return pd.Series(rma.rolling_mean())
test_df["SMA_30"] = sma_rma()

[sma_rma] Time elapsed: 0.000304 seconds
[sma_rma] Peak memory: 15.69 KB


In [14]:
rma.window = 90
test_df["SMA_90"] = sma_rma()
rma.window = 180
test_df["SMA_180"] = sma_rma()

[sma_rma] Time elapsed: 0.000240 seconds
[sma_rma] Peak memory: 14.31 KB
[sma_rma] Time elapsed: 0.000123 seconds
[sma_rma] Peak memory: 14.14 KB


In [15]:
sma_rma()

[sma_rma] Time elapsed: 0.000195 seconds
[sma_rma] Peak memory: 14.14 KB


0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
          ...    
188    219.366667
189    219.527778
190    219.672222
191    219.866667
192    220.055556
Length: 193, dtype: float64

In [16]:
test_df.tail(10)

,Date,Ticker,Open,High,Low,Close,Volume,SMA_30,SMA_90,SMA_180
183,2025-09-26,AAPL,254,258,254,255,46076300,236.900000,217.411111,218.844444
184,2025-09-29,AAPL,255,255,253,254,40127700,237.633333,217.933333,218.905556
185,2025-09-30,AAPL,255,256,253,255,37704300,238.433333,218.522222,219.005556
186,2025-10-01,AAPL,255,259,255,255,48713900,239.233333,219.122222,219.122222
187,2025-10-02,AAPL,257,258,254,257,42630200,240.266667,219.811111,219.255556
188,2025-10-03,AAPL,255,259,254,258,49155600,241.366667,220.455556,219.366667
189,2025-10-06,AAPL,258,259,255,257,44664100,242.333333,221.088889,219.527778
190,2025-10-07,AAPL,257,257,255,256,31923700,243.300000,221.711111,219.672222
191,2025-10-08,AAPL,257,258,256,258,6547574,244.266667,222.344444,219.866667
192,2025-10-08,AAPL,257,258,256,258,7671583,245.200000,222.966667,220.055556


In [17]:
uda = UpwardsDownwardsArray(test_df['Close'])

In [18]:
@measure_time_and_space
def create_run_group():
    """
    Calculate the difference between consecutive 'close' prices in-place.
    """
    return uda.create_run_group()

@measure_time_and_space
def create_run_group_naive():
    """
    Calculate the difference between consecutive 'close' prices in-place.
    """
    return uda.create_run_group_naive()

test_df["Run_Group"] = create_run_group()
test_df["Run_Group"] = create_run_group_naive()



[create_run_group] Time elapsed: 0.000031 seconds
[create_run_group] Peak memory: 0.08 KB
[create_run_group_naive] Time elapsed: 0.000020 seconds
[create_run_group_naive] Peak memory: 1.59 KB


In [19]:
test_df

,Date,Ticker,Open,High,Low,Close,Volume,SMA_30,SMA_90,SMA_180,Run_Group
0,2025-01-02,AAPL,249,249,242,244,55740700,NaN,NaN,NaN,NaN
1,2025-01-03,AAPL,243,244,242,243,40244100,NaN,NaN,NaN,0.0
2,2025-01-06,AAPL,244,247,243,245,45045600,NaN,NaN,NaN,1.0
3,2025-01-07,AAPL,243,246,241,242,40856000,NaN,NaN,NaN,-1.0
4,2025-01-08,AAPL,242,244,240,243,37628900,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...
188,2025-10-03,AAPL,255,259,254,258,49155600,241.366667,220.455556,219.366667,0.0
189,2025-10-06,AAPL,258,259,255,257,44664100,242.333333,221.088889,219.527778,-1.0
190,2025-10-07,AAPL,257,257,255,256,31923700,243.300000,221.711111,219.672222,0.0
191,2025-10-08,AAPL,257,258,256,258,6547574,244.266667,222.344444,219.866667,1.0


## Start of actual Feature Engineering


In [22]:
rolling_windows = [30, 90, 180]  # 1, 3, 6 months roughly

@measure_time_and_space
def process_group(group: pd.DataFrame) -> pd.DataFrame:
    close_vals = group["Close"].tolist()
    

    # 1️⃣ Compute multiple rolling means efficiently
    for w in rolling_windows:
        if len(close_vals) >= w:
            ma_vals = RollingMeanArray(close_vals, w).rolling_mean()
        else:
            ma_vals = [math.nan] * len(close_vals)
        group[f"MA_{w}"] = ma_vals

    # 2️⃣ Compute price differences -> Run Group
    group["Run_Group"] = UpwardsDownwardsArray(close_vals).create_run_group()

    return group


# Apply per company
final_df = combined_df.groupby("Ticker", group_keys=False, sort=False).apply(process_group)


[process_group] Time elapsed: 0.001596 seconds
[process_group] Peak memory: 29.28 KB
[process_group] Time elapsed: 0.001474 seconds
[process_group] Peak memory: 28.86 KB
[process_group] Time elapsed: 0.001297 seconds
[process_group] Peak memory: 28.92 KB
[process_group] Time elapsed: 0.001352 seconds
[process_group] Peak memory: 35.15 KB
[process_group] Time elapsed: 0.001540 seconds
[process_group] Peak memory: 35.24 KB
[process_group] Time elapsed: 0.001518 seconds
[process_group] Peak memory: 28.95 KB
[process_group] Time elapsed: 0.001364 seconds
[process_group] Peak memory: 34.40 KB


In [24]:
final_df['Daily_Return'] = (final_df.groupby("Ticker")['Close'].pct_change() * 100).round(2)

In [ ]:
final_df['Daily_Return'] = final_df['Close'].pct_change() * 100
final_df['Daily_Return'] = final_df['Daily_Return'].round(2)


In [25]:
final_df.tail(30)

,Date,Ticker,Open,High,Low,Close,Volume,MA_30,MA_90,MA_180,Run_Group,Daily_Return
1318,2025-08-27,TSLA,352,355,349,350,65519000,327.100000,317.855556,NaN,-1.0,-0.57
1319,2025-08-28,TSLA,351,354,340,346,67903200,328.000000,319.166667,NaN,-1.0,-1.14
1320,2025-08-29,TSLA,347,349,332,334,81145700,328.133333,320.233333,NaN,-1.0,-3.47
1321,2025-09-02,TSLA,328,333,326,329,58392000,328.166667,321.100000,NaN,-1.0,-1.50
1322,2025-09-03,TSLA,335,343,329,334,88733300,328.233333,321.922222,NaN,1.0,1.52
1323,2025-09-04,TSLA,336,339,331,339,60711000,328.433333,322.522222,NaN,1.0,1.50
1324,2025-09-05,TSLA,348,356,345,351,108989800,329.966667,323.244444,NaN,1.0,3.54
1325,2025-09-08,TSLA,355,358,345,346,75208300,330.966667,323.844444,NaN,-1.0,-1.42
1326,2025-09-09,TSLA,348,351,344,347,53816000,331.666667,324.566667,NaN,1.0,0.29
1327,2025-09-10,TSLA,351,356,346,348,72121700,332.566667,325.311111,NaN,1.0,0.29


In [27]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1348 entries, 0 to 1347
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          1348 non-null   datetime64[ns]
 1   Ticker        1348 non-null   object        
 2   Open          1348 non-null   int64         
 3   High          1348 non-null   int64         
 4   Low           1348 non-null   int64         
 5   Close         1348 non-null   int64         
 6   Volume        1348 non-null   int64         
 7   MA_30         1145 non-null   float64       
 8   MA_90         725 non-null    float64       
 9   MA_180        95 non-null     float64       
 10  Run_Group     1341 non-null   float64       
 11  Daily_Return  1341 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(5), object(1)
memory usage: 136.9+ KB


In [28]:
final_df.to_csv("mag7_stocks.csv", index=False)